In [30]:
%pip install -q gliner pandas pyarrow tqdm

import os
import torch
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT = Path("/content/beautyReview")
DRIVE_ROOT = Path("/content/drive/MyDrive/beautyReview")
INPUT_PATH = DRIVE_ROOT / "processed/reviews_sample_200.csv"
OUTPUT_ROOT = DRIVE_ROOT / "encoded"

assert PROJECT.exists(), "GitHub 프로젝트가 없습니다."
assert INPUT_PATH.exists(), "리뷰 샘플 CSV가 없습니다."
assert torch.cuda.is_available(), "Colab GPU 연결을 확인하세요."

os.chdir(PROJECT)

print("GPU:", torch.cuda.get_device_name(0))
print("프로젝트:", PROJECT)
print("입력 데이터:", INPUT_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
프로젝트: /content/beautyReview
입력 데이터: /content/drive/MyDrive/beautyReview/processed/reviews_sample_200.csv


In [31]:
from config.category_config import get_gliner_labels

df = pd.read_csv(INPUT_PATH)

required_columns = [
    "review_id",
    "product_category",
    "text"
]

assert all(c in df.columns for c in required_columns)
assert df["review_id"].is_unique, "review_id 중복 발견"

df["text"] = df["text"].fillna("").astype(str)

print("전체 리뷰:", len(df))
print("카테고리 개수:", df["product_category"].nunique())
display(df["product_category"].value_counts())

print("\nbath_body 기본 라벨:")
print(get_gliner_labels("bath_body"))


전체 리뷰: 2200
카테고리 개수: 11


,count
product_category,
bath_body,200
beauty_tools,200
fragrance,200
grooming,200
hair_accessories,200
haircare,200
makeup,200
nailcare,200
oralcare,200



bath_body 기본 라벨:
['product material', 'product defect', 'product scent', 'physical discomfort', 'skin type', 'cosmetic ingredient', 'skin reaction']


In [35]:
import json
from pathlib import Path

CONFIG_DIR = Path("/content/beautyReview/config/experiments")

for version in ["v1", "v2", "v3"]:
    candidates = [
        CONFIG_DIR / f"gliner_v{version[-1]}.json",
        CONFIG_DIR / f"gliner{version[-1]}.json"
    ]

    for path in candidates:
        if not path.exists():
            continue

        try:
            with open(path, encoding="utf-8-sig") as f:
                data = json.load(f)

            print(f"✅ 정상: {path.name}")

        except json.JSONDecodeError as e:
            print(f"❌ 오류: {path.name}")
            print(f"   위치: {e.lineno}행 {e.colno}열")
            print(f"   원인: {e.msg}")
            print(f"   파일 크기: {path.stat().st_size} bytes")

❌ 오류: gliner1.json
   위치: 1행 1열
   원인: Expecting value
   파일 크기: 0 bytes
❌ 오류: gliner2.json
   위치: 1행 1열
   원인: Expecting value
   파일 크기: 0 bytes
✅ 정상: gliner3.json


In [38]:
import json
import pandas as pd
from pathlib import Path

CONFIG_DIR = PROJECT / "config/experiments"

CONFIG_FILES = {
    "v1": "gliner_v1.json",
    "v2": "gliner_v2.json",
    "v3": "gliner_v3.json"
}

def load_config(version):
    path = CONFIG_DIR / CONFIG_FILES[version]

    if not path.exists():
        raise FileNotFoundError(f"설정 파일 없음: {path}")

    try:
        with open(path, "r", encoding="utf-8-sig") as f:
            config = json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"{path.name} JSON 형식 오류: "
            f"{e.lineno}행 {e.colno}열 - {e.msg}"
        ) from e

    if config["version"] != version:
        raise ValueError(
            f"{path.name}: 버전이 일치하지 않습니다."
        )

    required = [
        "model",
        "batch_size",
        "inference_threshold",
        "label_overrides"
    ]

    for key in required:
        if key not in config:
            raise KeyError(f"{path.name}: {key} 누락")

    print(f"✅ {version}: {path.name} 불러오기 완료")
    return config


configs = {
    version: load_config(version)
    for version in ["v1", "v2", "v3"]
}

display(pd.DataFrame([
    {
        "version": v,
        "model": c["model"],
        "batch_size": c["batch_size"],
        "threshold": c["inference_threshold"],
        "label_changes": len(c["label_overrides"])
    }
    for v, c in configs.items()
]))


FileNotFoundError: 설정 파일 없음: /content/beautyReview/config/experiments/gliner_v1.json

In [ ]:
def get_labels_and_mapping(category, config):
    base_labels = get_gliner_labels(category)
    overrides = config.get("label_overrides", {})

    labels = []
    mapping = {}

    for original in base_labels:
        model_label = overrides.get(original, original)

        labels.append(model_label)

        # 기존 짧은 Label을 표준 식별자로 사용
        normalized = original.lower().replace(" ", "_")

        mapping[model_label] = normalized

    if len(labels) != len(set(labels)):
        raise ValueError(
            f"{category}: 변경된 Label이 중복됩니다."
        )

    return labels, mapping


for version, config in configs.items():
    labels, mapping = get_labels_and_mapping(
        "bath_body", config
    )
    print(f"\n{version}:")
    for label in labels:
        print(" ", label, "→", mapping[label])

In [ ]:
from gliner import GLiNER
from tqdm.auto import tqdm
import gc

ENTITY_COLUMNS = [
    "review_id", "product_category",
    "entity_text", "entity_label", "model_label",
    "score", "start", "end",
    "config_version", "model_name"
]

STATUS_COLUMNS = [
    "review_id", "product_category",
    "entity_count", "config_version"
]


def run_experiment(version, config, df):
    model_name = config["model"]
    batch_size = int(config["batch_size"])
    threshold = float(config["inference_threshold"])

    print(f"\n===== {version} 시작 =====")
    print("모델:", model_name)
    print("Threshold:", threshold)

    model = GLiNER.from_pretrained(
        model_name
    ).to("cuda")

    model.eval()

    entity_rows = []
    status_rows = []

    try:
        for category, group in df.groupby(
            "product_category", sort=True
        ):
            labels, mapping = get_labels_and_mapping(
                category, config
            )

            records = group.to_dict("records")

            for i in tqdm(
                range(0, len(records), batch_size),
                desc=f"{version}: {category}"
            ):
                batch = records[i:i + batch_size]
                texts = [str(r["text"]) for r in batch]

                with torch.inference_mode():
                    predictions = model.inference(
                        texts,
                        labels,
                        batch_size=batch_size,
                        threshold=threshold
                    )

                assert len(predictions) == len(batch)

                for review, entities in zip(
                    batch, predictions
                ):
                    status_rows.append({
                        "review_id": review["review_id"],
                        "product_category": category,
                        "entity_count": len(entities),
                        "config_version": version
                    })

                    for e in entities:
                        model_label = e["label"]

                        entity_rows.append({
                            "review_id": review["review_id"],
                            "product_category": category,
                            "entity_text": e["text"],
                            "entity_label": mapping[model_label],
                            "model_label": model_label,
                            "score": float(e["score"]),
                            "start": e.get("start"),
                            "end": e.get("end"),
                            "config_version": version,
                            "model_name": model_name
                        })

    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

    entities_df = pd.DataFrame(
        entity_rows, columns=ENTITY_COLUMNS
    )

    status_df = pd.DataFrame(
        status_rows, columns=STATUS_COLUMNS
    )

    return entities_df, status_df


In [ ]:

FORCE_RERUN = False

for version in ["v1", "v2", "v3"]:
    config = configs[version]
    output = OUTPUT_ROOT / f"gliner_{version}"

    entity_file = output / "entities.parquet"
    status_file = output / "status.parquet"
    config_file = output / "experiment.json"

    can_skip = False

    if (
        not FORCE_RERUN
        and entity_file.exists()
        and status_file.exists()
        and config_file.exists()
    ):
        with open(config_file, encoding="utf-8") as f:
            saved_config = json.load(f)

        saved_status = pd.read_parquet(status_file)

        can_skip = (
            saved_config == config
            and len(saved_status) == len(df)
            and set(saved_status["review_id"])
                == set(df["review_id"])
        )

    if can_skip:
        print(f"\n{version}: 기존 결과 재사용")
        continue

    entities, status = run_experiment(
        version, config, df
    )

    assert len(status) == len(df), (
        f"{version}: 일부 리뷰 누락!"
    )

    save_experiment(
        version, config, entities, status
    )

    del entities, status
    gc.collect()

print("\n모든 실험 완료!")


In [ ]:

summary_rows = []
all_results = []

for version in ["v1", "v2", "v3"]:
    folder = OUTPUT_ROOT / f"gliner_{version}"

    entities = pd.read_parquet(
        folder / "entities.parquet"
    )
    status = pd.read_parquet(
        folder / "status.parquet"
    )

    summary_rows.append({
        "version": version,
        "model": configs[version]["model"],
        "reviews": len(status),
        "reviews_with_entities":
            int((status["entity_count"] > 0).sum()),
        "entity_count": len(entities),
        "mean_score":
            entities["score"].mean()
            if not entities.empty else None,
        "entities_per_review":
            len(entities) / len(status)
    })

    all_results.append(entities)

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    OUTPUT_ROOT / "experiment_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(summary_df)

comparison_df = pd.concat(
    all_results, ignore_index=True
)

label_counts = pd.crosstab(
    comparison_df["entity_label"],
    comparison_df["config_version"]
)

display(label_counts)


In [ ]:

target_ids = [
    "R0007560",
    "R0018430",
    "R0020146",
    "R0032597",
    "R0042046",
    "R0046094",
    "R0056704",
    "R0238918"
]

comparison = comparison_df[
    comparison_df["review_id"].isin(target_ids)
].copy()

display(
    comparison[
        [
            "review_id",
            "config_version",
            "entity_text",
            "entity_label",
            "score"
        ]
    ].sort_values(
        ["review_id", "config_version", "score"],
        ascending=[True, True, False]
    )
)
